# HuggingFace smolagents Practice
This notebook demonstrates how to use the `smolagents` framework to create a basic agent, define custom tools, integrate MCP tools, and observe the agent's behavior.

We will use an **Azure OpenAI** deployment (`gpt-5.4-mini` by default) for this tutorial.

## 1. Install and Import Dependencies
First, we install `smolagents` with the `openai` extra (client for OpenAI-compatible APIs such as Azure OpenAI) and the `toolkit` extra (default web search and webpage tools). We also install `python-dotenv` to load credentials from a `.env` file.

In [1]:
%pip install -q "smolagents[openai,toolkit]>=1.26" python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from smolagents import CodeAgent, OpenAIModel, tool

c:\Users\jelisejs.bumazkins\Documents\GitHub\Bootcamp_Accenture_JB\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configure the Azure OpenAI Model
Azure OpenAI's v1 API (`https://<resource>.openai.azure.com/openai/v1/`) is OpenAI-compatible, so we use smolagents' `OpenAIModel` and point its `api_base` at the Azure endpoint. The `model_id` is your Azure **deployment name** (e.g. `gpt-5.4-mini` or `gpt-5.6-luna`).

**Note:** Store your credentials in a `.env` file (in this folder or any parent folder). If a value is missing, the cell below prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [3]:
from smolagents import CodeAgent, OpenAIModel, tool

model = OpenAIModel(
    model_id="qwen3.5:2b-q4_K_M",
    api_base="http://127.0.0.1:11434/v1",
    api_key="ollama",
    reasoning_effort="none",
    max_tokens=512,
)

print("Local Qwen configured")

Local Qwen configured


## 3. Create a Basic CodeAgent
We'll start by creating a simple `CodeAgent` without any custom tools. This agent can write and execute Python code to solve problems.

In [4]:
basic_agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=2
)

print("Basic CodeAgent initialized successfully")

Basic CodeAgent initialized successfully


## 4. Define a Custom Tool
We can extend the agent's capabilities by defining custom tools. Here, we create a simple calculator tool using the `@tool` decorator.

In [5]:
@tool
def calculate_tip(bill_amount: float, tip_percentage: float = 15.0) -> str:
    """
    Calculates the tip amount and total bill.

    Args:
        bill_amount: The total cost of the meal.
        tip_percentage: The percentage of tip to leave (default is 15%).
    """
    tip = bill_amount * (tip_percentage / 100)
    total = bill_amount + tip
    return f"Tip: ${tip:.2f}, Total: ${total:.2f}"

# Test the tool directly
print(calculate_tip(100, 20))

Tip: $20.00, Total: $120.00


## 5. Integrate an MCP Tool
The Model Context Protocol (MCP) allows agents to connect to external tools standardized across different systems.

In `smolagents`, you can load tools from an MCP server with `MCPClient`.
```python
# Example of loading tools from an MCP server (requires `pip install "smolagents[mcp]"` and a running MCP server)
# from smolagents import MCPClient
# with MCPClient({"url": "http://localhost:8000/mcp", "transport": "streamable-http"}, structured_output=True) as mcp_tools:
#     agent = CodeAgent(tools=mcp_tools, model=model)
```

For this practice, we will simulate an MCP tool that retrieves system status.

In [6]:
@tool
def get_system_status(system_id: str) -> str:
    """
    Retrieves the status of a specific system.

    Args:
        system_id: The ID of the system to check (e.g., 'server-01').
    """
    # Simulated MCP response
    statuses = {"server-01": "Online", "server-02": "Maintenance", "db-prod": "Active"}
    return f"System {system_id} status: {statuses.get(system_id, 'Unknown')}"

## 6. Enable Observability
Observability is crucial for understanding how your agent makes decisions. `smolagents` provides built-in logging.
We can also use `GradioUI` for an interactive trace, but here we will rely on the console output which shows the agent's "thought" process.

When running the agent, `smolagents` prints the steps:
1.  **Thought**: The agent's reasoning.
2.  **Code**: The code generated to call a tool.
3.  **Observation**: The result from the tool.

In [7]:
# No specific code needed to enable basic logging, it's on by default in the run output.
# However, we can inspect the agent's logs after execution.
print("Observability is enabled by default. Watch the output of the next cell.")

Observability is enabled by default. Watch the output of the next cell.


## 7. Run the Agent with Tools
Now we combine everything. We create a new `CodeAgent` equipped with both our `calculate_tip` tool and the `get_system_status` tool.
We will give it a task that requires using both.

In [8]:
final_agent = CodeAgent(
    tools=[calculate_tip, get_system_status],
    model=model,
    max_steps=3,
)

response = final_agent.run(
    "Use calculate_tip to calculate a 20% tip on $150. "
    "Use get_system_status to check server-01. "
    "Report both results."
)

print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Use calculate_tip to calculate a 20% tip on $150. Use get_system_status to check server-01. Report both         │
│ results.                                                                                                        │
│                                                                                                                 │
╰─ OpenAIModel - qwen3.5:2b-q4_K_M ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Calculate tip: 20% of 150 is 30.                                                                               
  # Format returned as string, e.g., "45.00"                                                                       
  calculate                                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'calculate' due to: InterpreterError: The variable `calculate` is not defined.

[Step 1: Duration 123.96 seconds| Input tokens: 2,287 | Output tokens: 512]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (6 outputs left out)

[Step 3: Duration 33.90 seconds| Input tokens: 8,552 | Output tokens: 970]

Reached max steps.

[Step 4: Duration 50.74 seconds| Input tokens: 10,015 | Output tokens: 1,100]

To complete the task, I will use the `calculate_tip` tool to determine the 20% tip amount on $150 and then call `get_system_status` to verify the status of server-01.

First, I calculate the tip using the bill amount of $150 and a tip percentage of 20%. The result is $30.00.
Next, I check the system status for server-01, which reports it as "Online".

Here are the results:
**Tip:** $30.00
**Server Status (server-01):** Online


## Conclusion
You have successfully created a `smolagents` agent powered by Azure OpenAI, defined custom tools, simulated an MCP tool integration, and observed the agent's execution flow.
Experiment with adding more tools or changing the model to see how the agent adapts!

## 8. Challenge: Build a Smart Home Controller
Now it's your turn! Apply what you've learned to build a simple smart home controller.

**Your Task:**
1.  Create a custom tool `get_room_temperature(room: str)` that returns a simulated temperature (e.g., "18°C").
2.  Create a custom tool `set_thermostat(room: str, target_temp: int)` that simulates setting the temperature and returns a confirmation message.
3.  Instantiate a `CodeAgent` with these tools and the Azure OpenAI model.
4.  Run the agent with a query: *"Check the temperature in the Living Room. If it's below 20°C, set the thermostat to 22°C."*

In [9]:

# Challenge 8: Smart Home Controller

from smolagents import CodeAgent, tool

ROOM_TEMPERATURES = {
    "living room": 18,
    "bedroom": 21,
    "kitchen": 20,
}

THERMOSTAT_SETTINGS = {}

@tool
def get_room_temperature(room: str) -> int:
    """Return the room temperature as an integer in Celsius.

    Args:
        room: Name of the room, e.g. Living Room.
    """
    return ROOM_TEMPERATURES[room.strip().lower()]


@tool
def set_thermostat(room: str, target_temp: int) -> str:
    """Set the simulated thermostat temperature.

    Args:
        room: Name of the room.
        target_temp: Target temperature in Celsius.
    """
    key = room.strip().lower()
    THERMOSTAT_SETTINGS[key] = target_temp
    return f"Thermostat in {room} set to {target_temp}°C."


# Check the tool directly
print("Current temperature:", get_room_temperature("Living Room"))

# Let the agent make the decision
smart_home_agent = CodeAgent(
    tools=[get_room_temperature, set_thermostat],
    model=model,
    max_steps=3,
)

answer = smart_home_agent.run(
    "Call get_room_temperature(room='Living Room'). "
    "It returns an integer. If it is below 20, "
    "call set_thermostat(room='Living Room', target_temp=22). "
    "Then return a short confirmation."
)

print("Agent response:", answer)
print("Thermostat state:", THERMOSTAT_SETTINGS)

assert THERMOSTAT_SETTINGS.get("living room") == 22
print("Challenge 8 PASSED")


Current temperature: 18


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Call get_room_temperature(room='Living Room'). It returns an integer. If it is below 20, call                   │
│ set_thermostat(room='Living Room', target_temp=22). Then return a short confirmation.                           │
│                                                                                                                 │
╰─ OpenAIModel - qwen3.5:2b-q4_K_M ───────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  temperature = get_room_temperature(room='Living Room')                                                           
  print(f"Temperature is {temperature}.")                                                                          
                                                                                                                   
  if temperature < 20:                                                                                             
      set_thermostat(room='Living Room', target_temp=22)                                                           
      print("Thermostat has been set to 22 degrees.")                                                              
  else:                                                                                                            
      print("Room temperature is acceptable (above 20 degrees).")                                                  
                                                                                                                   
  final_answer("The temperature check passed or was set successfully.")                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Temperature is 18.
Thermostat has been set to 22 degrees.

Final answer: The temperature check passed or was set successfully.

[Step 1: Duration 49.07 seconds| Input tokens: 2,277 | Output tokens: 172]

Agent response: The temperature check passed or was set successfully.
Thermostat state: {'living room': 22}
Challenge 8 PASSED
